# 02 — Model Training, Evaluation & Feature Importance

**Student Burnout & Dropout Risk Dataset**

This notebook:
1. Loads and preprocesses the dataset (missing-value imputation, encoding, scaling)
2. Engineers composite features
3. Trains **Dropout Risk** (binary) and **Burnout Level** (multiclass) classifiers:
   - Logistic Regression
   - Random Forest (GridSearchCV)
   - XGBoost (hyperparameter tuning)
4. Compares models with metrics, ROC curves, and confusion matrices
5. Runs SHAP analysis on the best model
6. Exports best models and metadata for downstream use

---
## 1. Setup & Data Preparation

In [ ]:
# -- Imports -----------------------------------------------------------------
import warnings, os, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, precision_score, recall_score, roc_curve, auc
)
from xgboost import XGBClassifier
import shap

# -- Style ------------------------------------------------------------------
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
BINARY_PALETTE = {"Yes": "#e74c3c", "No": "#2ecc71"}
MULTI_PALETTE  = ["#2ecc71", "#f39c12", "#e74c3c"]  # Low / Medium / High
CMAP           = "YlOrRd"

PROJECT_DIR  = os.getcwd() if os.path.exists(os.path.join(os.getcwd(), "data.csv")) else os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PATH    = os.path.join(PROJECT_DIR, "data.csv")
MODELS_DIR   = os.path.join(PROJECT_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

print(f"Project dir : {PROJECT_DIR}")
print(f"Data path   : {DATA_PATH}")
print(f"Models dir  : {MODELS_DIR}")

In [ ]:
# -- Load raw data -----------------------------------------------------------
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# -- Missing values before imputation ----------------------------------------
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0].to_string())
print(f"\nTotal missing: {missing.sum()}")

In [ ]:
# -- Impute missing values ---------------------------------------------------
numeric_cols     = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

# Remove targets and ID
categorical_cols = [c for c in categorical_cols if c not in ("Burnout_Level", "Dropout_Risk", "Student_ID")]

for col in numeric_cols:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"  Filled {col} (numeric) with median = {median_val:.2f}")

for col in categorical_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f"  Filled {col} (categorical) with mode = '{mode_val}'")

print(f"\nRemaining missing: {df.isnull().sum().sum()}")

In [ ]:
# -- Encode categoricals ------------------------------------------------------
label_encoders = {}

# Binary targets
le_dropout = LabelEncoder()
df["Dropout_Risk_enc"] = le_dropout.fit_transform(df["Dropout_Risk"])
label_encoders["Dropout_Risk"] = le_dropout

le_burnout = LabelEncoder()
df["Burnout_Level_enc"] = le_burnout.fit_transform(df["Burnout_Level"])
label_encoders["Burnout_Level"] = le_burnout

# Feature categoricals
for col in categorical_cols:
    le = LabelEncoder()
    df[col + "_enc"] = le.fit_transform(df[col])
    label_encoders[col] = le
    print(f"  {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

print("\nEncoding complete.")

In [ ]:
# -- Build feature matrix (before derived features) -------------------------
drop_cols = ["Student_ID", "Burnout_Risk", "Dropout_Risk", "Burnout_Level"]
feature_cols_raw = [c for c in df.columns if c not in drop_cols
                    and c not in ("Burnout_Level_enc",)
                    and c != "Student_ID_enc"]

print(f"Feature columns ({len(feature_cols_raw)}):")
for i, c in enumerate(feature_cols_raw):
    print(f"  {i+1:2d}. {c}")

In [ ]:
# -- Train / Test Split (80/20 stratified) -----------------------------------
X_raw    = df[feature_cols_raw].values
y_bin    = df["Dropout_Risk_enc"].values
y_multi  = df["Burnout_Level_enc"].values

X_train_raw, X_test_raw, y_bin_train, y_bin_test, y_multi_train, y_multi_test = \
    train_test_split(X_raw, y_bin, y_multi, test_size=0.2, random_state=42, stratify=y_bin)

print(f"Train: {X_train_raw.shape[0]}  |  Test: {X_test_raw.shape[0]}")
print(f"Binary target dist  - train: {np.bincount(y_bin_train)}  test: {np.bincount(y_bin_test)}")
print(f"Multi  target dist  - train: {np.bincount(y_multi_train)}  test: {np.bincount(y_multi_test)}")

---
## 2. Feature Engineering

Create three composite domain features:
- **academic_engagement_score** - weighted blend of attendance, study hours, GPA
- **lifestyle_risk_score** - high screen time, low sleep, low exercise
- **financial_burden_score** - financial stress inversely weighted by family support

In [ ]:
# -- Derived features --------------------------------------------------------
def add_derived_features(dataframe):
    """Add composite scores to the dataframe."""
    # Academic engagement: normalise each component to [0,1] then average
    att = (dataframe["Attendance_Percent"] - dataframe["Attendance_Percent"].min()) / \
          (dataframe["Attendance_Percent"].max() - dataframe["Attendance_Percent"].min() + 1e-9)
    hrs = (dataframe["Study_Hours_Per_Day"] - dataframe["Study_Hours_Per_Day"].min()) / \
          (dataframe["Study_Hours_Per_Day"].max() - dataframe["Study_Hours_Per_Day"].min() + 1e-9)
    gpa = (dataframe["Previous_GPA"] - dataframe["Previous_GPA"].min()) / \
          (dataframe["Previous_GPA"].max() - dataframe["Previous_GPA"].min() + 1e-9)
    dataframe["academic_engagement_score"] = (att + hrs + gpa) / 3.0

    # Lifestyle risk: high screen_time & low sleep & low exercise = higher risk
    scr = (dataframe["Screen_Time_Hours"] - dataframe["Screen_Time_Hours"].min()) / \
          (dataframe["Screen_Time_Hours"].max() - dataframe["Screen_Time_Hours"].min() + 1e-9)
    slp = 1.0 - (dataframe["Sleep_Hours"] - dataframe["Sleep_Hours"].min()) / \
          (dataframe["Sleep_Hours"].max() - dataframe["Sleep_Hours"].min() + 1e-9)
    exr = 1.0 - (dataframe["Exercise_Freq_Per_Week"] - dataframe["Exercise_Freq_Per_Week"].min()) / \
          (dataframe["Exercise_Freq_Per_Week"].max() - dataframe["Exercise_Freq_Per_Week"].min() + 1e-9)
    dataframe["lifestyle_risk_score"] = (scr + slp + exr) / 3.0

    # Financial burden: financial stress normalised, mitigated by family support
    fst = (dataframe["Financial_Stress_Score"] - dataframe["Financial_Stress_Score"].min()) / \
          (dataframe["Financial_Stress_Score"].max() - dataframe["Financial_Stress_Score"].min() + 1e-9)
    fsu = (dataframe["Family_Support_Score"] - dataframe["Family_Support_Score"].min()) / \
          (dataframe["Family_Support_Score"].max() - dataframe["Family_Support_Score"].min() + 1e-9)
    dataframe["financial_burden_score"] = fst * (1.0 - 0.4 * fsu)

    return dataframe

df = add_derived_features(df)
print("Derived features added:")
for feat in ["academic_engagement_score", "lifestyle_risk_score", "financial_burden_score"]:
    print(f"  {feat}  mean={df[feat].mean():.3f}  std={df[feat].std():.3f}")

In [ ]:
# -- Distribution of derived features ----------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, feat, color in zip(axes,
    ["academic_engagement_score", "lifestyle_risk_score", "financial_burden_score"],
    ["#3498db", "#e74c3c", "#9b59b6"]):
    sns.histplot(df[feat], bins=30, kde=True, color=color, ax=ax)
    ax.set_title(feat.replace("_", " ").title(), fontweight="bold")
    ax.set_xlabel("")
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, "derived_features_dist.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: derived_features_dist.png")

In [ ]:
# -- Rebuild feature matrix WITH derived features ----------------------------
derived_cols = ["academic_engagement_score", "lifestyle_risk_score", "financial_burden_score"]
feature_cols_final = feature_cols_raw + derived_cols

X       = df[feature_cols_final].values
y_bin   = df["Dropout_Risk_enc"].values
y_multi = df["Burnout_Level_enc"].values

X_train, X_test, y_bin_train, y_bin_test, y_multi_train, y_multi_test = \
    train_test_split(X, y_bin, y_multi, test_size=0.2, random_state=42, stratify=y_bin)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Final feature count: {X.shape[1]}  (was {len(feature_cols_raw)}, +3 derived)")
print(f"Train shape: {X_train_sc.shape}  |  Test shape: {X_test_sc.shape}")

---
## 3. Model Training - Dropout Risk (Binary)

Three classifiers: **Logistic Regression**, **Random Forest** (GridSearchCV), and **XGBoost** (hyperparameter tuning).

In [ ]:
# -- Helper functions -------------------------------------------------------
def evaluate_binary(clf, name, X_tr, y_tr, X_te, y_te):
    """Evaluate a binary classifier and plot confusion matrix."""
    y_pred = clf.predict(X_te)
    report = classification_report(y_te, y_pred, target_names=le_dropout.classes_)
    acc = accuracy_score(y_te, y_pred)
    print(f"{'='*50}")
    print(f"  {name}  -  Accuracy: {acc:.4f}")
    print(f"{'='*50}")
    print(report)

    # Confusion matrix heatmap
    cm = confusion_matrix(y_te, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap=CMAP, cbar=False,
                xticklabels=le_dropout.classes_, yticklabels=le_dropout.classes_, ax=ax)
    ax.set_xlabel("Predicted", fontweight="bold")
    ax.set_ylabel("Actual", fontweight="bold")
    ax.set_title(f"{name} - Confusion Matrix", fontweight="bold")
    plt.tight_layout()
    plt.show()

    metrics = {
        "Accuracy":  accuracy_score(y_te, y_pred),
        "F1":        f1_score(y_te, y_pred, average="weighted"),
        "Precision": precision_score(y_te, y_pred, average="weighted"),
        "Recall":    recall_score(y_te, y_pred, average="weighted"),
    }
    return {"name": name, "model": clf, "metrics": metrics, "y_pred": y_pred}


def evaluate_multiclass(clf, name, X_tr, y_tr, X_te, y_te):
    """Evaluate a multiclass classifier and plot confusion matrix."""
    y_pred = clf.predict(X_te)
    report = classification_report(y_te, y_pred, target_names=le_burnout.classes_)
    acc = accuracy_score(y_te, y_pred)
    print(f"{'='*50}")
    print(f"  {name}  -  Accuracy: {acc:.4f}")
    print(f"{'='*50}")
    print(report)

    cm = confusion_matrix(y_te, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap=CMAP, cbar=False,
                xticklabels=le_burnout.classes_, yticklabels=le_burnout.classes_, ax=ax)
    ax.set_xlabel("Predicted", fontweight="bold")
    ax.set_ylabel("Actual", fontweight="bold")
    ax.set_title(f"{name} - Confusion Matrix", fontweight="bold")
    plt.tight_layout()
    plt.show()

    metrics = {
        "Accuracy":  accuracy_score(y_te, y_pred),
        "F1":        f1_score(y_te, y_pred, average="weighted"),
        "Precision": precision_score(y_te, y_pred, average="weighted"),
        "Recall":    recall_score(y_te, y_pred, average="weighted"),
    }
    return {"name": name, "model": clf, "metrics": metrics, "y_pred": y_pred}

print("Helper functions defined.")

### 3.1 Logistic Regression - Binary

In [ ]:
# -- Logistic Regression (binary) -------------------------------------------
lr_bin = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
lr_bin.fit(X_train_sc, y_bin_train)
lr_bin_result = evaluate_binary(lr_bin, "Logistic Regression",
                                 X_train_sc, y_bin_train, X_test_sc, y_bin_test)

### 3.2 Random Forest - Binary (GridSearchCV)

In [ ]:
# -- Random Forest (binary) - GridSearchCV ----------------------------------
rf_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
}
rf_bin_cv = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight="balanced"),
    rf_param_grid, cv=5, scoring="f1_weighted", n_jobs=-1, verbose=0
)
rf_bin_cv.fit(X_train_sc, y_bin_train)
print(f"Best RF params (binary): {rf_bin_cv.best_params_}")
print(f"Best CV F1:              {rf_bin_cv.best_score_:.4f}")
rf_bin_result = evaluate_binary(rf_bin_cv.best_estimator_,
                                 "Random Forest (GridSearchCV)",
                                 X_train_sc, y_bin_train, X_test_sc, y_bin_test)

### 3.3 XGBoost - Binary (Hyperparameter Tuning)

In [ ]:
# -- XGBoost (binary) - hyperparameter tuning --------------------------------
xgb_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}
xgb_bin_cv = GridSearchCV(
    XGBClassifier(
        random_state=42, use_label_encoder=False, eval_metric="logloss",
        scale_pos_weight=(y_bin_train == 0).sum() / max((y_bin_train == 1).sum(), 1)
    ),
    xgb_param_grid, cv=5, scoring="f1_weighted", n_jobs=-1, verbose=0
)
xgb_bin_cv.fit(X_train_sc, y_bin_train)
print(f"Best XGB params (binary): {xgb_bin_cv.best_params_}")
print(f"Best CV F1:               {xgb_bin_cv.best_score_:.4f}")
xgb_bin_result = evaluate_binary(xgb_bin_cv.best_estimator_,
                                  "XGBoost (Tuned)",
                                  X_train_sc, y_bin_train, X_test_sc, y_bin_test)

In [ ]:
# -- Store binary results for later comparison -------------------------------
binary_results = [lr_bin_result, rf_bin_result, xgb_bin_result]
print("Binary model results stored.")

---
## 4. Model Training - Burnout Level (Multiclass)

Same three classifiers, now predicting **Low / Medium / High** burnout.

### 4.1 Logistic Regression - Multiclass

In [ ]:
# -- Logistic Regression (multiclass) ----------------------------------------
lr_mc = LogisticRegression(max_iter=1000, random_state=42,
                           multi_class="multinomial", class_weight="balanced")
lr_mc.fit(X_train_sc, y_multi_train)
lr_mc_result = evaluate_multiclass(lr_mc, "Logistic Regression",
                                    X_train_sc, y_multi_train, X_test_sc, y_multi_test)

### 4.2 Random Forest - Multiclass

In [ ]:
# -- Random Forest (multiclass) - GridSearchCV -------------------------------
rf_mc_cv = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight="balanced"),
    rf_param_grid, cv=5, scoring="f1_weighted", n_jobs=-1, verbose=0
)
rf_mc_cv.fit(X_train_sc, y_multi_train)
print(f"Best RF params (multiclass): {rf_mc_cv.best_params_}")
print(f"Best CV F1:                  {rf_mc_cv.best_score_:.4f}")
rf_mc_result = evaluate_multiclass(rf_mc_cv.best_estimator_,
                                    "Random Forest (GridSearchCV)",
                                    X_train_sc, y_multi_train, X_test_sc, y_multi_test)

### 4.3 XGBoost - Multiclass

In [ ]:
# -- XGBoost (multiclass) - hyperparameter tuning ----------------------------
xgb_mc_cv = GridSearchCV(
    XGBClassifier(random_state=42, use_label_encoder=False, eval_metric="mlogloss"),
    xgb_param_grid, cv=5, scoring="f1_weighted", n_jobs=-1, verbose=0
)
xgb_mc_cv.fit(X_train_sc, y_multi_train)
print(f"Best XGB params (multiclass): {xgb_mc_cv.best_params_}")
print(f"Best CV F1:                   {xgb_mc_cv.best_score_:.4f}")
xgb_mc_result = evaluate_multiclass(xgb_mc_cv.best_estimator_,
                                     "XGBoost (Tuned)",
                                     X_train_sc, y_multi_train, X_test_sc, y_multi_test)

In [ ]:
# -- Store multiclass results ------------------------------------------------
multiclass_results = [lr_mc_result, rf_mc_result, xgb_mc_result]
print("Multiclass model results stored.")

---
## 5. Model Comparison

### 5.1 Metric Comparison (Bar Charts)

In [ ]:
# -- Side-by-side bar charts for binary & multiclass -------------------------
metric_names = ["Accuracy", "F1", "Precision", "Recall"]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, results, title in zip(axes,
        [binary_results, multiclass_results],
        ["Dropout Risk (Binary)", "Burnout Level (Multiclass)"]):
    names = [r["name"] for r in results]
    data  = {m: [r["metrics"][m] for r in results] for m in metric_names}
    x = np.arange(len(names))
    width = 0.18
    colors = ["#3498db", "#2ecc71", "#f39c12", "#e74c3c"]
    for i, m in enumerate(metric_names):
        bars = ax.bar(x + i * width, data[m], width, label=m,
                      color=colors[i], edgecolor="white")
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                    f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=7)
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(names, rotation=15, ha="right", fontsize=9)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Score", fontweight="bold")
    ax.set_title(title, fontweight="bold", fontsize=13)
    ax.legend(loc="lower right", fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, "model_comparison_bars.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: model_comparison_bars.png")

### 5.2 ROC Curves - Binary Dropout Risk

In [ ]:
# -- ROC curves for binary classifiers ---------------------------------------
fig, ax = plt.subplots(figsize=(8, 6))
colors = ["#3498db", "#2ecc71", "#e74c3c"]
y_bin_test_bin = label_binarize(y_bin_test, classes=[0, 1]).ravel()

for result, color in zip(binary_results, colors):
    clf = result["model"]
    if hasattr(clf, "predict_proba"):
        y_score = clf.predict_proba(X_test_sc)[:, 1]
    else:
        y_score = clf.decision_function(X_test_sc)
    fpr, tpr, _ = roc_curve(y_bin_test_bin, y_score)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f'{result["name"]} (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5, label="Random")
ax.set_xlabel("False Positive Rate", fontweight="bold")
ax.set_ylabel("True Positive Rate", fontweight="bold")
ax.set_title("ROC Curves - Dropout Risk (Binary)", fontweight="bold", fontsize=13)
ax.legend(loc="lower right", fontsize=9)
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, "roc_curves_binary.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: roc_curves_binary.png")

### 5.3 Best Model Summary

In [ ]:
# -- Identify best model per task --------------------------------------------
best_bin = max(binary_results,  key=lambda r: r["metrics"]["F1"])
best_mc  = max(multiclass_results, key=lambda r: r["metrics"]["F1"])

print("=" * 60)
print("  BEST MODEL - DROPOUT RISK (Binary)")
print("=" * 60)
print(f"  Model : {best_bin['name']}")
for k, v in best_bin["metrics"].items():
    print(f"  {k:10s}: {v:.4f}")

print()
print("=" * 60)
print("  BEST MODEL - BURNOUT LEVEL (Multiclass)")
print("=" * 60)
print(f"  Model : {best_mc['name']}")
for k, v in best_mc["metrics"].items():
    print(f"  {k:10s}: {v:.4f}")

---
## 6. SHAP Feature Importance Analysis

Using **TreeExplainer** on the best XGBoost model to explain feature contributions.

In [ ]:
# -- SHAP on best binary XGBoost --------------------------------------------
best_xgb_bin = xgb_bin_cv.best_estimator_

print("Computing SHAP values (binary - Dropout Risk) ...")
explainer_bin = shap.TreeExplainer(best_xgb_bin)
shap_values_bin = explainer_bin.shap_values(X_test_sc)

feature_names = feature_cols_final
print(f"SHAP values shape: {np.array(shap_values_bin).shape}")

In [ ]:
# -- Beeswarm plot ----------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values_bin, X_test_sc, feature_names=feature_names,
                  show=False, max_display=20)
plt.title("SHAP Beeswarm - XGBoost (Dropout Risk)", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, "shap_beeswarm_binary.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: shap_beeswarm_binary.png")

In [ ]:
# -- Identify top 3 features by mean |SHAP| ----------------------------------
mean_abs_shap = np.abs(shap_values_bin).mean(axis=0)
top3_idx = np.argsort(mean_abs_shap)[::-1][:3]
top3_features = [feature_names[i] for i in top3_idx]

print("Top 3 features by mean |SHAP|:")
for rank, (idx, feat) in enumerate(zip(top3_idx, top3_features), 1):
    print(f"  {rank}. {feat}  (mean |SHAP| = {mean_abs_shap[idx]:.4f})")

In [ ]:
# -- Dependence plots for top 3 features -------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, idx, feat in zip(axes, top3_idx, top3_features):
    plt.sca(ax)
    shap.dependence_plot(idx, shap_values_bin, X_test_sc,
                         feature_names=feature_names, show=False, ax=ax)
    ax.set_title(f"SHAP Dependence - {feat}", fontweight="bold", fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, "shap_dependence_top3.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: shap_dependence_top3.png")

In [ ]:
# -- Bar plot of feature importances -----------------------------------------
shap_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Mean_SHAP": mean_abs_shap
}).sort_values("Mean_SHAP", ascending=True)

fig, ax = plt.subplots(figsize=(8, 10))
colors_bar = plt.cm.RdYlBu_r(np.linspace(0.2, 0.8, len(shap_importance_df)))
ax.barh(shap_importance_df["Feature"], shap_importance_df["Mean_SHAP"], color=colors_bar)
ax.set_xlabel("Mean |SHAP Value|", fontweight="bold")
ax.set_title("Feature Importance (SHAP) - XGBoost Dropout Risk",
             fontweight="bold", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, "shap_feature_importance_bar.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved: shap_feature_importance_bar.png")

In [ ]:
# -- Save feature importance to CSV ------------------------------------------
shap_importance_df_sorted = shap_importance_df.sort_values("Mean_SHAP", ascending=False)
csv_path = os.path.join(MODELS_DIR, "feature_importance.csv")
shap_importance_df_sorted.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")
display(shap_importance_df_sorted.head(10))

---
## 7. Model Export & Metadata

Saving:
- Best binary & multiclass models
- Scaler, label encoders
- Feature metadata for inference pipelines

In [ ]:
# -- Save best models -------------------------------------------------------
joblib.dump(best_bin["model"], os.path.join(MODELS_DIR, "best_dropout_risk_model.joblib"))
print(f"Saved: best_dropout_risk_model.joblib ({best_bin['name']})")

joblib.dump(best_mc["model"], os.path.join(MODELS_DIR, "best_burnout_level_model.joblib"))
print(f"Saved: best_burnout_level_model.joblib ({best_mc['name']})")

joblib.dump(scaler, os.path.join(MODELS_DIR, "scaler.joblib"))
print("Saved: scaler.joblib")

In [ ]:
# -- Create feature_metadata.joblib -----------------------------------------
feature_metadata = {
    "feature_names": feature_cols_final,
    "n_features": len(feature_cols_final),
    "numeric_columns": numeric_cols,
    "categorical_columns": categorical_cols,
    "label_encoders": label_encoders,
    "dropout_risk_classes": le_dropout.classes_.tolist(),
    "burnout_level_classes": le_burnout.classes_.tolist(),
    "best_binary_model": best_bin["name"],
    "best_multiclass_model": best_mc["name"],
    "binary_metrics": best_bin["metrics"],
    "multiclass_metrics": best_mc["metrics"],
    "derived_features": {
        "academic_engagement_score": "Mean of normalised attendance, study hours, GPA",
        "lifestyle_risk_score": "Mean of normalised screen time, inverse sleep, inverse exercise",
        "financial_burden_score": "Financial stress normalised, mitigated by family support",
    },
}

meta_path = os.path.join(MODELS_DIR, "feature_metadata.joblib")
joblib.dump(feature_metadata, meta_path)
print(f"Saved: {meta_path}")

# Print metadata summary (exclude label_encoders for readability)
meta_summary = {k: v for k, v in feature_metadata.items() if k != "label_encoders"}
print(json.dumps(meta_summary, indent=2, default=str))

In [ ]:
# -- Final summary of saved artefacts ----------------------------------------
print("=" * 60)
print("  SAVED ARTIFACTS")
print("=" * 60)
for f in sorted(os.listdir(MODELS_DIR)):
    fpath = os.path.join(MODELS_DIR, f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {f:45s}  {size_kb:7.1f} KB")
print("=" * 60)
print("\nNotebook complete - all models trained, evaluated, and exported.")